# Pipeline de embeddings: MACCROBAT2020 → split → PubMedBERT → classificadores

Este notebook só **chama** os módulos de `models/pre_processamento/`:

| Módulo | Responsabilidade |
|---|---|
| `brat_reader.py` | lê os pares `.txt` / `.ann` (entidades e relações) |
| `split_cases.py` | divide treino / validação / teste **por caso** e salva a divisão |
| `embeddings_bert.py` | `BertEmbedder`: texto cru → embeddings por subtoken (com posições em caracteres) |
| `build_datasets.py` | monta os datasets de **nós** e de **arestas** (`X`, `y`) e guarda em cache |
| `evaluation.py` | F1 por entidade |

Para encaixar outro classificador basta colocá-lo no dicionário da seção 4: qualquer objeto com `fit(X, y)` e `predict(X)` serve.

In [2]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()                       # este notebook fica em notebooks/
sys.path.insert(0, str(ROOT / "models" / "pre_processamento"))

import brat_reader, split_cases, build_datasets
from embeddings_bert import BertEmbedder

DATA_DIR = ROOT / "data" / "treinamento" / "MACCROBAT2020"
OUT_DIR = ROOT / "data" / "treinamento" / "embeddings"

c:\Users\hidal\Documents\GitHub\Diplotokens-P1\project 1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Dados e divisão

In [3]:
docs = brat_reader.load_docs(DATA_DIR)
print(len(docs), "casos | entidades com posição desalinhada:", brat_reader.check_offsets(docs))

splits = split_cases.get_splits([d.id for d in docs], OUT_DIR / "splits.json", seed=42)
print({k: len(v) for k, v in splits.items()})

200 casos | entidades com posição desalinhada: 0
{'train': 140, 'val': 30, 'test': 30}


## 2. Embeddings (o BERT roda uma vez; depois vem do cache)

`mask_digits=True` troca cada dígito por `0` e permite testar o efeito de esconder os valores das medidas - se for realizar lembrar de extrair antes as medidas usando o regex do P2

In [4]:
embedder = BertEmbedder(mask_digits=False)
build_datasets.build_token_dataset(docs, splits, embedder, OUT_DIR)

train = build_datasets.load_token_dataset(OUT_DIR, "train", embedder.name)
val = build_datasets.load_token_dataset(OUT_DIR, "val", embedder.name)
test = build_datasets.load_token_dataset(OUT_DIR, "test", embedder.name)
print(train["X"].shape, val["X"].shape, test["X"].shape)

c:\Users\hidal\Documents\GitHub\Diplotokens-P1\project 1\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hidal\.cache\huggingface\hub\models--microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\hidal\Documents\GitHub\Diplotokens-

[train] 140 casos -> train_BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.npz
[val] 30 casos -> val_BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.npz
[test] 30 casos -> test_BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.npz
(82488, 768) (15947, 768) (16362, 768)
